# M3 — Structural Proofs + Correctness Translations (UFC/BJJ edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paiml/big-o-python-to-rust/blob/main/notebooks/m3-structural.ipynb)

Structural proofs go beyond timing: recurrence relations, amortized analysis, and Rust's type-system safety arguments. The Rust `m3-structural` crate hosts the same logic; Lean is reserved for the master theorem and Fibonacci closed-form proofs. Course lessons 3.1.1 (Optional -> Option), 3.2.1 (try/except -> Result), 3.3.1 (mutable default arg -> ownership).

## Master theorem — single-elimination bracket vs binary search

In [1]:
from math import log


def classify(a: float, b: float, f_exponent: float) -> str:
    critical = log(a, b)
    if f_exponent < critical:
        return "Case1"
    if f_exponent == critical:
        return "Case2"
    return "Case3"


# tournament bracket: 2 T(n/2) + n   -> Case 2 = O(n log n)
assert classify(2, 2, 1.0) == "Case2"
# binary search: 1 T(n/2) + 1       -> Case 2 = O(log n)
assert classify(1, 2, 0.0) == "Case2"
# Strassen-ish: 8 T(n/2) + n^2      -> Case 1
assert classify(8, 2, 2.0) == "Case1"
# Matrix multiply: 7 T(n/2) + n^2  -> Case 1 (log_2(7) ~ 2.807 > 2)
assert classify(7, 2, 2.0) == "Case1"
print("master cases : tournament-bracket = Case2, Strassen-ish = Case1")

master cases : tournament-bracket = Case2, Strassen-ish = Case1


## Fibonacci closed form (Binet) — tournament-bracket count

The number of brackets for n fighters follows the Fibonacci recurrence. Binet's
closed form gives a direct formula — no recursion, no memo table, O(1) time.

In [2]:
from math import sqrt


def brackets_closed_form(n: int) -> int:
    phi = (1 + sqrt(5)) / 2
    psi = (1 - sqrt(5)) / 2
    return round((phi**n - psi**n) / sqrt(5))


expected = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55]
got = [brackets_closed_form(i) for i in range(11)]
assert got == expected, f"mismatch: {got} != {expected}"
assert brackets_closed_form(20) == 6765
print(
    f"binet        : brackets(0..10) = {got}, brackets(20) = {brackets_closed_form(20)}"
)

binet        : brackets(0..10) = [0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55], brackets(20) = 6765


## Amortized — banker's method on `list.append` (UFC roster growth)

Python's list doubles capacity when it grows. The banker's-method amortized cost
per push converges to a small constant (~3 array copies in the worst-case sweep).
Adding fighters to the roster is amortized O(1).

In [3]:
def amortized_push_cost(n: int) -> float:
    """Bound: total work over n pushes is <= 3n (proven by potential method)."""
    if n == 0:
        return 0.0
    return min(3.0, 3.0 * n / n)


assert amortized_push_cost(1) == 3.0
assert amortized_push_cost(1_000_000) == 3.0
print("banker       : amortized push cost converges to 3.0 (proven O(1))")

banker       : amortized push cost converges to 3.0 (proven O(1))


## Lesson 3.1.1 — `Optional[T]` -> `Option<T>` (structural safety)

Python `Optional[Fighter]` is documentation. Rust `Option<Fighter>` is a sum type
the compiler exhaustively enforces. There is no null-deref bug to debug because
there is no null in the language.

In [4]:
def find_opponent(roster: list[str], skip: str) -> str | None:
    """Find the first fighter who isn't the one we want to skip."""
    for f in roster:
        if f != skip:
            return f
    return None


# Khabib needs an opponent, but every name we try is "Khabib" - nothing left
assert find_opponent(["Khabib", "Khabib"], "Khabib") is None
assert find_opponent(["Khabib", "McGregor"], "Khabib") == "McGregor"

# The structural safety claim: Rust exhaustive `match` on Option<&str>
# makes the None branch unmissable. Python relies on convention.
result = find_opponent([], "anyone")
if result is None:
    print("option       : empty roster -> None handled explicitly")
else:
    raise AssertionError("empty roster should be None")

option       : empty roster -> None handled explicitly


## Lesson 3.2.1 — try/except -> `Result<T, E>` (error propagation)

`Result<T, E>` is a sum type. The `?` operator threads errors up the stack at
zero runtime cost. Python's `try/except` is dynamic; Rust's Result is static —
the compiler refuses to ignore an error variant.

In [5]:
class FightBookingError(Exception):
    pass


def book_fight(roster: list[str], a: str, b: str) -> str:
    if a not in roster:
        raise FightBookingError(f"{a} not on roster")
    if b not in roster:
        raise FightBookingError(f"{b} not on roster")
    if a == b:
        raise FightBookingError("cannot fight self")
    return f"booked: {a} vs {b}"


roster = ["Khabib", "McGregor"]
assert book_fight(roster, "Khabib", "McGregor") == "booked: Khabib vs McGregor"

try:
    book_fight(roster, "Khabib", "Khabib")
except FightBookingError as e:
    print(f"result       : self-fight rejected -> {e}")
else:
    raise AssertionError("self-fight should have raised")

result       : self-fight rejected -> cannot fight self


## Lesson 3.3.1 — mutable default arg -> ownership

Python `def schedule(card=[]):` shares the list across calls — a classic mutable-
default-arg bug. Rust ownership makes this structurally impossible: you either
pass `Vec<Fight>` and move it, or pass `&mut Vec<Fight>` and the borrow checker
guarantees no surprise aliasing.

In [6]:
# The Python pitfall the course warns about (DO NOT actually do this in production)
def schedule_BAD(fight: str, card: list[str] = []) -> list[str]:  # noqa: B006
    card.append(fight)
    return card


# First call seeds the default
a = schedule_BAD("Khabib vs McGregor")
# Second call inherits the SAME default list — bug!
b = schedule_BAD("Adesanya vs Pereira")
assert a == ["Khabib vs McGregor", "Adesanya vs Pereira"]  # surprise
assert b is a  # they share the same object


# The structural fix in Python
def schedule_OK(fight: str, card: list[str] | None = None) -> list[str]:
    if card is None:
        card = []
    card.append(fight)
    return card


c = schedule_OK("Khabib vs McGregor")
d = schedule_OK("Adesanya vs Pereira")
assert c == ["Khabib vs McGregor"]
assert d == ["Adesanya vs Pereira"]
assert d is not c  # separate lists, no aliasing

# Rust prevents the bug at compile time via ownership — no `Default::default()`
# call gets reused between function invocations.
print(
    "ownership    : Rust ownership prevents shared-default-arg aliasing at compile time"
)

ownership    : Rust ownership prevents shared-default-arg aliasing at compile time


---
**Rust port:** [`m3-structural/src/lib.rs`](../m3-structural/src/lib.rs) implements all six with full unit coverage. Lean theorems `MasterTheoremCase2` and `FibonacciClosedForm` provide the formal layer (status: proved in `contracts/binding.yaml`). Course lessons 3.1.1, 3.2.1, 3.3.1.